# G4-conv - cross-architecture hub transfer

ConvNeXt is already cached, so no re-encoding. **Check native R@1 against the published 93-96% band before trusting any transfer number** - the spectrum gate confirmed the construction, not the retrieval protocol.

## Storage

In [ ]:
import os
from pathlib import Path
STORAGE   = "drive"
DRIVE_DIR = "/content/drive/MyDrive/convergence_experiment"
LOCAL_DIR = "./convergence_data"
try:
    import google.colab                 # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if STORAGE == "env":
    assert os.environ.get("DATA_DIR"), "STORAGE='env' but DATA_DIR unset"
elif STORAGE == "drive" and IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    os.environ["DATA_DIR"] = DRIVE_DIR
else:
    os.environ["DATA_DIR"] = str(Path(LOCAL_DIR).resolve())
DATA_DIR = Path(os.environ["DATA_DIR"])
print("DATA_DIR:", DATA_DIR)

## Experiment

In [ ]:
# ==========================================================
# REQUIRES G0_hub_rebuild TO HAVE PASSED ITS VALIDATION GATE.
# The original hub artifacts were never persisted. G0 rebuilds them
# and refuses to write anything unless the reconstruction reproduces
# the published singular values. If G0 stopped, do not run this.
# ==========================================================
# G4-conv — cross-ARCHITECTURE hub transfer: a convolutional encoder.
# Closes the last C.13.2 gap. Paste as ONE cell after G4.
#
# PRE-REGISTERED, before running. Write the prediction down, then run.
#
#   Every encoder in the project so far is a transformer. G4 established that
#   lineage does not matter (SigLIP 2, no DINOv2 ancestry, 94.2% of native,
#   inside the 93.8-96.5% within-family band). It did NOT establish that
#   ARCHITECTURE does not matter. ConvNeXt is the test: convolutional
#   inductive bias, hierarchical local receptive fields, no patch tokens,
#   no self-attention anywhere.
#
#   PREDICTION: ConvNeXt lands INSIDE or just below the 93.8-96.5% band.
#   Rationale: C.13.5 found objective (+53x) > lineage (+44x) > modality
#   (+29x). Architecture was never in that ranking. If the thesis is about
#   what training induces rather than what the network is made of, a
#   convnet trained on the same objective family should behave like a
#   sibling.
#
#   WHAT FALSIFIES IT: below ~85%, with the random-map control still at
#   chance and rows/dim still above 5. That would put architecture into the
#   causal ranking and bound the claim to transformers - a real limit,
#   worth more than a confirmation.
#
#   Either way this is ONE number and the answer is reportable. Do not
#   sweep, do not tune. Fit, read, write it down.
# ==========================================================
import os
import numpy as np
from pathlib import Path
DATA_DIR = Path(os.environ.get("DATA_DIR", "."))

HUB_NPZ = DATA_DIR / "hub_rebuilt.npz"    # written by G0 - run G0 first
# Everything else comes out of that file: the aligned raw_* spaces, the hub
# coordinates, and the split. Only ConvNeXt is loaded separately.
# ----------------------------------------------------------------------

N_TRAIN, N_EVAL = 8533, 1000
BAND = (0.938, 0.965)   # within-family zero-shot band from C.13.2
SEED = 0

In [ ]:
# ---------- 1. load the cached ConvNeXt embeddings ----------
# Already encoded on 17 Aug by the same e1_img_ckpt pipeline as every other
# encoder, so row order matches the other spaces BY CONSTRUCTION - no fresh
# forward pass, no image ordering to reconstruct, no alignment risk.
#
# Note the two protocol facts to state alongside the result:
#   * "-22k"    ImageNet-22k supervised pretraining, not self-supervised
#   * "_native" the same pooling convention as the SigLIP 2 cache that
#               produced the 94.2% cross-lineage number, so this is
#               protocol-matched to the comparison it will be read against.

CONVNEXT_NPZ = DATA_DIR / "e1_img_ckpt_convnext-base-224-22k_native.npz"
assert CONVNEXT_NPZ.exists(), f"not found: {CONVNEXT_NPZ}"


def load_embedding(path, n_expected=N_TRAIN + N_EVAL):
    """Pull the embedding matrix without assuming a key name.

    Takes the widest 2-D float array with the expected row count. If more
    than one qualifies the file is ambiguous and this stops rather than
    guessing - picking the wrong array here would produce a plausible
    number with no meaning.
    """
    z = np.load(path, allow_pickle=True)
    cands = []
    for k in z.files:
        a = z[k]
        if (getattr(a, "ndim", 0) == 2 and a.shape[0] == n_expected
                and np.issubdtype(a.dtype, np.floating)):
            cands.append((k, a))
    print(f"  {path.name}")
    for k in z.files:
        a = z[k]
        print(f"     {k:28s} {getattr(a, 'shape', '-')}  {getattr(a, 'dtype', '-')}")
    if not cands:
        raise KeyError(f"no 2-D float array with {n_expected} rows in {path.name}")
    if len(cands) > 1:
        widest = max(cands, key=lambda t: t[1].shape[1])
        print(f"  NOTE: {len(cands)} candidate arrays; taking widest "
              f"'{widest[0]}' {widest[1].shape}. Confirm this is the "
              f"embedding before reading any result.")
        return widest[1].astype(np.float64)
    print(f"  using '{cands[0][0]}' {cands[0][1].shape}")
    return cands[0][1].astype(np.float64)


X_conv = load_embedding(CONVNEXT_NPZ)
X_conv = X_conv / (np.linalg.norm(X_conv, axis=1, keepdims=True) + 1e-8)

print(f"\nConvNeXt ambient width: {X_conv.shape[1]}")
print(f"rows per input dimension: {N_TRAIN / X_conv.shape[1]:.1f}  "
      f"(pre-registered floor: 5)")
assert N_TRAIN / X_conv.shape[1] >= 5, "under-powered - do not read the result"

In [ ]:
# ---------- 2. sanity: is this space healthy before we map it? ----------
def eff_rank(M):
    ev = np.clip(np.linalg.eigvalsh(np.cov(M.T.astype(np.float64))), 0, None)
    return float(ev.sum() ** 2 / (ev ** 2).sum())

sub = X_conv[np.random.default_rng(SEED).choice(len(X_conv), 1500, False)]
pair_cos = float((sub @ sub.T)[np.triu_indices(1500, 1)].mean())
print(f"mean pair-cosine: {pair_cos:+.3f}   "
      f"(collapse flag is > 0.30 - see C.13.10)")
print(f"effective rank:   {eff_rank(X_conv):.1f} of {X_conv.shape[1]}")
if pair_cos > 0.30:
    print("  FLAG: degenerate frame. Any transfer gain here is the C.11 "
          "rescue, not architecture-independence. Whiten and re-read.")

In [ ]:
# ---------- 3. align ConvNeXt to the hub's row order ----------
# G0 sorted the shared rows by image id. ConvNeXt's cache carries its own
# keep array, so it must be put into the SAME order before anything is
# fitted. This is asserted on set equality, not assumed: a silent reorder
# here would misalign every row and produce a low transfer number that
# looks exactly like a real architecture effect.
hub = np.load(HUB_NPZ, allow_pickle=True)
enc_names = [str(x) for x in hub["encoder_names"]]
H_all = hub["H_all"]                       # [9533, 768] hub coordinates
N_TR = int(hub["n_train"])
N_EV = int(hub["n_eval"])
print(f"  hub coords {H_all.shape}, split {N_TR}/{N_EV}")
print(f"  encoders in hub: {', '.join(enc_names)}")

zc = np.load(CONVNEXT_NPZ, allow_pickle=True)
keep_conv = zc["keep"]
X_conv_raw = zc["img"].astype(np.float64)

# The two groups of caches use DIFFERENT id namespaces:
#   img_small / img_base / hub_ids : keep = positional indices 0..N-1,
#                                    i.e. arange - it carries no image
#                                    identity whatsoever
#   convnext / siglip              : keep = real COCO ids (9..46492)
# So an id join is impossible, and cross-group intersections are noise
# (convnext ∩ img_small = 1879 is just how many COCO ids fall below 9533).
#
# What IS informative: convnext ∩ siglip = 9533, identical sets. SigLIP is
# the encoder that produced the published 94.2%, so ConvNeXt shares its
# image set exactly and whatever worked for SigLIP works here unchanged.
#
# That leaves POSITIONAL identity as the only hypothesis: row i of ConvNeXt
# is row i of img_small, both written by the same loop over the same list.
# Plausible - and precisely the kind of assumption that fails silently, so
# it is TESTED rather than asserted.
X_conv = X_conv_raw
assert len(X_conv) == len(H_all), (
    f"row count differs: convnext {len(X_conv)} vs hub {len(H_all)}")

print("\n  ALIGNMENT TEST - positional identity, verified not assumed")
print("  If row i of ConvNeXt is row i of img_small, a ridge between them")
print("  predicts held-out rows well. Under any shuffle it predicts nothing.")

from sklearn.linear_model import Ridge as _R
_ref_raw = hub["raw_img_small"].astype(np.float64)
_ntr = int(hub["n_train"])


def _r2(src, tgt, ntr=_ntr):
    W = _R(alpha=1.0).fit(src[:ntr], tgt[:ntr]).coef_.T
    pred, t = src[ntr:] @ W, tgt[ntr:]
    return 1.0 - float(((t - pred) ** 2).sum() / (t ** 2).sum())


r2_aligned = _r2(_ref_raw, X_conv)
_perm = np.random.default_rng(SEED).permutation(len(X_conv))
r2_shuffled = _r2(_ref_raw, X_conv[_perm])

print(f"    img_small -> ConvNeXt, as-is      R^2 = {r2_aligned:+.3f}")
print(f"    img_small -> ConvNeXt, shuffled   R^2 = {r2_shuffled:+.3f}")
print(f"    separation                              {r2_aligned - r2_shuffled:+.3f}")

if r2_aligned - r2_shuffled < 0.05:
    print()
    print("  POSITIONAL ALIGNMENT FAILED. Predicting ConvNeXt from img_small")
    print("  is no better than predicting a shuffled ConvNeXt, so the rows do")
    print("  not correspond. The COCO ids ConvNeXt stores cannot be joined to")
    print("  anything, because the DINOv2 caches stored arange instead of ids")
    print("  - the mapping from row to image was never written down.")
    print()
    print("  STOPPING. A transfer number computed on misaligned rows would")
    print("  come out low and look exactly like a real architecture effect.")
    print("  That is the single most dangerous failure mode in this test.")
    print()
    print("  Recoverable only by re-encoding img_small with ids recorded, or")
    print("  by finding the original image list. Against the schedule, the")
    print("  cheaper answer is to report ConvNeXt as not run - the recommended")
    print("  next test, with the reason stated.")
    raise SystemExit("positional alignment failed")

print("  PASSED - rows correspond. Proceeding.")

In [ ]:
# ---------- 4. map ConvNeXt into the hub, fit a head, transfer it ----------
from sklearn.linear_model import Ridge

HUB_WIDTH = 512          # the operating point every headline number uses
H = H_all[:, :HUB_WIDTH]

# caption targets: G0 saved the aligned text spaces
T = hub["raw_txt_sbert"].astype(np.float64)
T = T / (np.linalg.norm(T, axis=1, keepdims=True) + 1e-8)
print(f"  caption targets (sbert, aligned): {T.shape}")

# ConvNeXt -> hub, one linear map, same as every other encoder
W_conv = Ridge(alpha=1.0).fit(X_conv[:N_TR], H[:N_TR]).coef_.T
H_conv = X_conv @ W_conv

# the head is trained on ONE encoder's hub coordinates and never sees ConvNeXt
ref = "img_small"
X_ref = hub[f"raw_{ref}"].astype(np.float64)
W_ref = Ridge(alpha=1.0).fit(X_ref[:N_TR], H[:N_TR]).coef_.T
H_ref = X_ref @ W_ref
W_head = Ridge(alpha=1.0).fit(H_ref[:N_TR], T[:N_TR]).coef_.T
print(f"  head fitted on {ref} hub coords, {HUB_WIDTH}-d -> {T.shape[1]}-d")


def recall_at_1(P, G):
    P = P / (np.linalg.norm(P, axis=1, keepdims=True) + 1e-8)
    return float(((P @ G.T).argmax(1) == np.arange(len(P))).mean())


r_ref = recall_at_1(H_ref[N_TR:] @ W_head, T[N_TR:])     # sanity: the source
r_zero = recall_at_1(H_conv[N_TR:] @ W_head, T[N_TR:])   # ConvNeXt, unseen

# native ceiling: a head fitted directly on ConvNeXt's own hub coords
W_native = Ridge(alpha=1.0).fit(H_conv[:N_TR], T[:N_TR]).coef_.T
r_native = recall_at_1(H_conv[N_TR:] @ W_native, T[N_TR:])

# control: random map in place of the learned one
rng = np.random.default_rng(SEED)
W_rand = rng.normal(size=W_conv.shape) / np.sqrt(W_conv.shape[0])
r_ctrl = recall_at_1((X_conv[N_TR:] @ W_rand) @ W_head, T[N_TR:])

pct = r_zero / r_native if r_native > 0 else float("nan")

In [ ]:
# ---------- 5. read it ----------
print("\n" + "=" * 58)
print("G4-conv - cross-ARCHITECTURE zero-shot component transfer")
print("=" * 58)
print(f"  source encoder R@1 ({ref})            {r_ref:.3f}")
print(f"  zero-shot R@1 (ConvNeXt, unseen)      {r_zero:.3f}")
print(f"  native R@1 (head fitted on ConvNeXt)  {r_native:.3f}")
print(f"  % of native                           {pct:.1%}")
print(f"  random-map control                    {r_ctrl:.3f}   "
      f"(chance = {1 / N_EV:.3f})")
print(f"  within-family band (C.13.2)           "
      f"{BAND[0]:.1%} - {BAND[1]:.1%}")

if r_ctrl > 0.01:
    print("\n  VERDICT: INVALID - the control is above chance. The head is "
          "carrying something on its own. Do not report the transfer "
          "number until this is understood.")
elif pct >= BAND[0]:
    print("\n  VERDICT: architecture does NOT bound the claim. A convnet "
          "sits inside the transformer band. The shared structure is a "
          "property of training, not of the computational substrate - "
          "which is the strongest form of the thesis the project can "
          "reach with these encoders.")
elif pct >= 0.85:
    print("\n  VERDICT: transfers, but BELOW the within-family band. "
          "Architecture costs something measurable. Report the number and "
          "the gap; do not round it into the band.")
else:
    print("\n  VERDICT: architecture BOUNDS the claim. Report as a limit: "
          "the hub carries transformers, and a convnet is a partial "
          "citizen. This is a more interesting result than a pass and it "
          "belongs in the abstract, not a footnote.")

print("\n  Scope to state with whichever verdict lands: one convnet, one "
      "image domain, ImageNet-22k SUPERVISED rather than self-supervised. "
      "ConvNeXt differs from DINOv2 in architecture AND objective, so a "
      "shortfall cannot be attributed to architecture alone. The honest "
      "framing is 'no encoder tested so far falls outside the band', not "
      "'architecture is irrelevant'.")